## Multi-Head Attention Plus Data Loading....



In [1]:
from importlib.metadata import version

print("Torch version:", version("torch"))

Torch version 2.9.0+cu126


The complete chapter code is located in [ch03.ipynb.](https://github.com/Sayanth789/My_ML_Projects/blob/main/LLMs-From-Scratch/chapt-3/01_main_chapter_code/ch03.ipynb/)

This notebook contains the main takeaway, multihead-attention implementation (plus the data loading pipeline from chapter 2)

## DataLoader from Chapter 2


In [10]:
import tiktoken
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader


class GPTDatasetV1(Dataset):
  def __init__(self, txt, tokenizer, max_length, stride):
    self.input_ids = []
    self.target_ids = []


    # Tokenize the entire text
    token_ids = tokenizer.encode(txt, allowed_special={"<|endoftext|>"})

    # Use  a sliding window to chunk the book into overalpping sequences of max_length
    for i in range(0, len(token_ids) - max_length, stride):
      input_chunk = token_ids[i: i + max_length]
      target_chunk = token_ids[i + 1: i + max_length + 1]
      self.input_ids.append(torch.tensor(input_chunk))
      self.target_ids.append(torch.tensor(target_chunk))

  def __len__(self):
    return len(self.input_ids)

  def __getitem__(self, idx):
    return self.input_ids[idx], self.target_ids[idx]


def create_dataloader(txt, batch_size=4, max_length=256, stride=128, shuffle=True):
  # Initalize the tokenizer
  tokenizer = tiktoken.get_encoding("gpt2")

  # Create dataset
  dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)

  # Create dataloder
  dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=shuffle)

  return dataloader

with open("small-text-sample.txt", "r", encoding="utf-8") as f:
  raw_text = f.read()


tokenizer = tiktoken.get_encoding("gpt2")
encodef_text = tokenizer.encode(raw_text)

vocab_size = 50257
output_dim = 256
max_len = 1024
context_length = max_len

token_embedding_layer  = nn.Embedding(vocab_size, output_dim)
pos_embedding_layer = torch.nn.Embedding(context_length, output_dim)


max_length = 4
dataloader = create_dataloader(raw_text, batch_size=8, max_length=max_length, stride=max_length)


In [11]:
for batch in dataloader:
    x, y = batch

    token_embeddings = token_embedding_layer(x)
    pos_embeddings = pos_embedding_layer(torch.arange(max_length))

    input_embeddings = token_embeddings + pos_embeddings

    break

In [13]:
print(input_embeddings.shape)

torch.Size([8, 4, 256])


## Mulit-head Attention from Chapter 3

#### Variant A: Simple implementation.


In [23]:
class CausalSelfAttention(nn.Module):

  def __init__(self, d_in, d_out, context_length, dropout, qkv_bias=False):
    super().__init__()
    self.d_out = d_out
    self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
    self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
    self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
    self.dropout = nn.Dropout(dropout)   # New
    self.register_buffer("mask", torch.triu(torch.ones(context_length, context_length)))

  def forward(self, x):
    b, n_tokens, d_in = x.shape   # New batch dimension b
    keys = self.W_key(x)
    values = self.W_value(x)
    queries = self.W_query(x)

    attn_scores = queries @ keys.transpose(1, 2)  # changed transpose
    attn_scores.masked_fill(
        self.mask.bool()[:n_tokens, :n_tokens], -torch.inf
    )
    attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
    attn_weights = self.dropout(attn_weights)  #new

    context_vec = attn_weights @ values
    return context_vec


class MultiHeadAttentionWrapper(nn.Module):
  def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
    super().__init__()
    self.heads = nn.ModuleList(
        [CausalSelfAttention(d_in, d_out, context_length, dropout, qkv_bias)
        for _ in range(num_heads)]
    )
    self.out_proj = nn.Linear(d_out * num_heads, d_out*num_heads)

  def forward(self, x):
    context_vec =  torch.cat([head(x) for head in self.heads], dim=-1)
    return self.out_proj(context_vec)



In [24]:
torch.manual_seed(123)

context_length = max_length
d_in = output_dim

num_heads=2
d_out = d_in // num_heads

mha = MultiHeadAttentionWrapper(d_in, d_out, context_length, 0.0, num_heads)

batch = input_embeddings
context_vecs = mha(batch)

print("context_vecs.shape:", context_vecs.shape)

context_vecs.shape: torch.Size([8, 4, 256])


### Variant B: Alternative implementation

In [37]:
class MultiHeadAttention(nn.Module):
  def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):

    super().__init__()
    assert d_out % num_heads == 0, "d_out must be divisible by num_heads"

    self.d_out = d_out
    self.num_heads = num_heads
    self.head_dim = d_out // num_heads  # Reduce the projection dim to match desired  output

    self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
    self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
    self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)

    self.out_proj = nn.Linear(d_out, d_out)  # linear layer to combine head outputs
    self.dropout = nn.Dropout(dropout)
    self.register_buffer("mask", torch.triu(torch.ones(context_length, context_length), diagonal=1))

  def forward(self, x):
    b, num_tokens , d_in = x.shape

    keys = self.W_key(x) # shape : (b, num_tokens, d_out)
    queries = self.W_query(x)
    values = self.W_value(x)


    # We implicitly split the matrix by adding a `num_heads` dimension
    # Unroll the last dim: (b, num_tokens, d_out) -> (b, num_tokens, num_heads, head_dim)
    keys = keys.view(b, num_tokens, self.num_heads, self.head_dim)
    values = values.view(b, num_tokens, self.num_heads, self.head_dim)
    queries = values.view(b, num_tokens, self.num_heads, self.head_dim)

    # Transpose last dim:  (b, num_tokens, num_heads, head_dim) -> (b, num_heads, num_tokens, head_dim)
    keys = keys.transpose(1, 2)
    queries = queries.transpose(1, 2)
    values = values.transpose(1, 2)


    # Compute the scaled dot-product attention (aka self-attention ) with a casual mask
    attn_scores = queries @ keys.transpose(2, 3)  # Dot-product for each head

    # Orginal mask truncated to the number of tokens and converted to boolean
    mask_bool = self.mask.bool()[:num_tokens, :num_tokens]

    # Use the mask ot fill attetntion scores
    attn_scores.masked_fill(mask_bool, -torch.inf)

    attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
    attn_weights = self.dropout(attn_weights)

    # Shape (b, num_tokens, num_heads, head_dim)
    context_vec = (attn_weights @ values).transpose(1, 2)

    # combine heads, num_tokens , num_heads, head_dim
    context_vec = context_vec.contiguous().view(b, num_tokens, self.d_out)

    context_vec = self.out_proj(context_vec)  # optional projection

    return context_vec





In [38]:

torch.manual_seed(123)

context_length = max_length
d_in = output_dim
d_out = d_in

mha = MultiHeadAttention(d_in, d_out, context_length, 0.0, num_heads=2)

batch = input_embeddings
context_vecs = mha(batch)

print("context_vecs.shape:", context_vecs.shape)

context_vecs.shape: torch.Size([8, 4, 256])
